# Temporal forecasting baselines

Create same-zone lags and compare hourly, daily, weekly, and zero predictions on September 2024.


In [ ]:
from pathlib import Path
import sys

# Run from either the project root or the notebooks directory.
project_root = Path.cwd() if (Path.cwd() / "notebooks").is_dir() else Path.cwd().parent
if str(project_root / "notebooks") not in sys.path:
    sys.path.insert(0, str(project_root / "notebooks"))

import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, root_mean_squared_error


In [ ]:
df = pd.read_parquet(str(project_root / "data/yellow_taxi_2024_aggreg.parquet"))


In [ ]:
X = df.copy()
y = X.trip_count

# Sort within each zone so shifts follow chronological hourly observations.
# Assign by index to preserve X's original row order and alignment with y.
zone_history = X.sort_values(["PULocationID", "tpep_pickup_datetime"])
zone_trip_counts = zone_history.groupby("PULocationID")["trip_count"]

X["lag_1h"] = zone_trip_counts.shift(1)
X["lag_24h"] = zone_trip_counts.shift(24)
X["lag_1w"] = zone_trip_counts.shift(7 * 24)


In [ ]:
X_train = X[X["tpep_pickup_datetime"].dt.month <= 8]
X_val = X[(X["tpep_pickup_datetime"].dt.month > 8)&(X["tpep_pickup_datetime"].dt.month <= 9)]
X_test = X[X["tpep_pickup_datetime"].dt.month > 9]
y_train = X_train.trip_count
y_val = X_val.trip_count
y_test = X_test.trip_count


#### Basic predictions : previous time


In [ ]:
y_pred_hour = X_val.lag_1h
y_pred_day = X_val.lag_24h
y_pred_week = X_val.lag_1w


In [ ]:
X_train


In [ ]:
mae_hour = mean_absolute_error(y_val, y_pred_hour)
rmse_hour = root_mean_squared_error(y_val, y_pred_hour)

mae_day = mean_absolute_error(y_val, y_pred_day)
rmse_day = root_mean_squared_error(y_val, y_pred_day)

mae_week = mean_absolute_error(y_val, y_pred_week)
rmse_week = root_mean_squared_error(y_val, y_pred_week)

print(f"Previous hour (lag_1h): MAE = {mae_hour:.4f}, RMSE = {rmse_hour:.4f}")
print(f"Previous day (lag_24h): MAE = {mae_day:.4f}, RMSE = {rmse_day:.4f}")
print(f"Previous week (lag_1w): MAE = {mae_week:.4f}, RMSE = {rmse_week:.4f}")


#### Zero predictors


In [ ]:
y_pred_zero = [0 for _ in range(len(y_val))]
mae_zero = mean_absolute_error(y_pred_zero, y_val)
rmse_zero = root_mean_squared_error(y_pred_zero, y_val)
print(f"Zero predictor: MAE = {mae_zero:.4f}, RMSE = {rmse_zero:.4f}")


In [ ]:
#Ranking the highest demand on an hourly average
rank = []
for z in X_train.PULocationID.unique():
    rank.append((z, X_train["trip_count"][X_train["PULocationID"] == z].mean()))

rank = sorted(rank, key = lambda x:x[1])
rank.reverse()

rank_zones = [rank[i][0] for i in range(len(rank))] #keep only the zone number

high_demand = rank_zones[:(263//3)]
mid_demand = rank_zones[(263//3):2*(263//3)]
low_demand = rank_zones[2*(263//3):]


In [ ]:
y_high = X_val.trip_count[X_val["PULocationID"].isin(high_demand)]
y_pred_high = X_val.lag_1w[X_val["PULocationID"].isin(high_demand)]

y_mid = X_val.trip_count[X_val.PULocationID.isin(mid_demand)]
y__pred_mid = X_val.lag_1w[X_val.PULocationID.isin(mid_demand)]

y_low = X_val.trip_count[X_val.PULocationID.isin(low_demand)]
y_pred_low = X_val.lag_1w[X_val.PULocationID.isin(low_demand)]

mae_high = mean_absolute_error(y_high, y_pred_high)
rmse_high = root_mean_squared_error(y_high, y_pred_high)

mae_mid = mean_absolute_error(y_mid, y__pred_mid)
rmse_mid = root_mean_squared_error(y_mid, y__pred_mid)

mae_low = mean_absolute_error(y_low, y_pred_low)
rmse_low = root_mean_squared_error(y_low, y_pred_low)

print("Validation results — previous week (lag_1w)")
print(f"High demand zones: MAE = {mae_high:.4f}, RMSE = {rmse_high:.4f}")
print(f"Mid demand zones:  MAE = {mae_mid:.4f}, RMSE = {rmse_mid:.4f}")
print(f"Low demand zones:  MAE = {mae_low:.4f}, RMSE = {rmse_low:.4f}")


In [ ]:
y_zero_high = np.zeros(len(y_high))
y_zero_mid = np.zeros(len(y_mid))
y_zero_low = np.zeros(len(y_low))

mae_zero_high = mean_absolute_error(y_high, y_zero_high)
rmse_zero_high = root_mean_squared_error(y_high, y_zero_high)

mae_zero_mid = mean_absolute_error(y_mid, y_zero_mid)
rmse_zero_mid = root_mean_squared_error(y_mid, y_zero_mid)

mae_zero_low = mean_absolute_error(y_low, y_zero_low)
rmse_zero_low = root_mean_squared_error(y_low, y_zero_low)

print("Validation results — zero predictions")
print(f"High demand zones: MAE = {mae_zero_high:.4f}, RMSE = {rmse_zero_high:.4f}")
print(f"Mid demand zones:  MAE = {mae_zero_mid:.4f}, RMSE = {rmse_zero_mid:.4f}")
print(f"Low demand zones:  MAE = {mae_zero_low:.4f}, RMSE = {rmse_zero_low:.4f}")


In [ ]:
X


In [ ]:
from pathlib import Path

lag_data_dir = Path("data") if Path("data").is_dir() else Path("../data")
lag_output_path = lag_data_dir / "yellow_taxi_2024_with_lags.parquet"
X.to_parquet(lag_output_path, index = False)
print(f"Saved {len(X):,} rows with lag features to {lag_output_path}")
